In [ ]:
import matplotlib
from matplotlib import font_manager as fm
from pathlib import Path

candidates = [
    "Noto Sans CJK SC", "Noto Sans CJK JP",
    "WenQuanYi Zen Hei", "WenQuanYi Micro Hei",
    "Source Han Sans SC", "Source Han Sans CN",
    "Droid Sans Fallback", "SimHei", "Microsoft YaHei", "Arial Unicode MS",
]
available = {f.name for f in fm.fontManager.ttflist}
chosen = next((n for n in candidates if n in available), None)
if chosen is None:
    for p in Path("/usr/share/fonts").rglob("*"):
        if p.suffix.lower() in {".ttf", ".ttc", ".otf"} and any(
            k in p.name.lower() for k in ["noto", "wqy", "sourcehan", "cjk", "hei", "droid"]
        ):
            fm.fontManager.addfont(str(p))
            chosen = fm.FontProperties(fname=str(p)).get_name()
            break
if chosen is None:
    print("这台机器没找到中文字体，标题先用英文")
else:
    matplotlib.rcParams["font.sans-serif"] = [chosen, "DejaVu Sans"]
    matplotlib.rcParams["font.family"] = "sans-serif"
    matplotlib.rcParams["axes.unicode_minus"] = False
    print("使用字体:", chosen)

# 05 Top10% 五批次重叠超额路径

每个交易日按预测超额 `pred` 取最高 10%，等权。资金拆成 5 批：t 收盘买入、t+5 收盘卖出，卖出所得再投入当天新篮子，批次独立复利。逐日盯市，组合超额 = 五批当天超额的净值加权平均，相对 688 平权。不满 5 批的日期不计入。

需要这一版的 `k_pictures_gray_body/results/pred_I20.parquet`（列名 `pred`），以及 `0914/data/processed/prices.parquet`。

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "change_rgb_i20"]
candidates.extend(cwd.parents)
for parent in [cwd, *cwd.parents]:
    candidates.append(parent / "change_rgb_i20")
seen = set()
root = None
for p in candidates:
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if (p / "helpers.py").exists() and (p / "config.yaml").exists() and (p / "train_lib.py").exists():
        text = (p / "config.yaml").read_text(encoding="utf-8")
        if "task: regression" in text:
            root = p
            break
if root is None:
    raise FileNotFoundError(
        f"找不到 change_rgb_i20（config.yaml 里要有 task: regression）。cwd={cwd}"
    )
missing = [name for name in ("train_lib.py", "backtest_lib.py") if not (root / name).exists()]
if missing:
    raise FileNotFoundError(f"{root} 缺少 {missing}。把这些文件和 helpers.py 放在同一目录。")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from helpers import notebook_bootstrap
from train_lib import (
    plot_random_groups, plot_top10_excess, plot_top10_gross_net, resolve_results_dir)
from backtest_lib import plot_decile_dashboard

ROOT, cfg, paths = notebook_bootstrap()
print("ROOT =", ROOT)
print("results =", resolve_results_dir(paths, cfg))
print("n_sleeves =", cfg.backtest.n_sleeves)
WINDOW_KEY = "I20"   # 或 "I60"

## 主结果：Top10% 五批次逐日盯市

In [ ]:
plot_top10_excess(cfg, paths, WINDOW_KEY)

## 十分位（同样 5 sleeve）

In [ ]:
plot_decile_dashboard(cfg, paths, WINDOW_KEY)

## 随机分组对照

In [ ]:
plot_random_groups(cfg, paths, window_key=WINDOW_KEY, n_groups=10)

## 成本：万五每边

In [ ]:
COST_BPS = 5
plot_top10_gross_net(cfg, paths, WINDOW_KEY, cost_bps_per_side=COST_BPS)